<a href="https://colab.research.google.com/github/Astro111000/Electropreorg_theozymes_rfd3/blob/main/Heavily_adapted_RFDiffusion3_%26_LigandMPNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Note that this is just a modified version of the already existing IPD foundry notebook here : https://github.com/RosettaCommons/foundry?tab=readme-ov-file#google-colab

## Issues regarding triton for RFD3 were fixed by downloading the checkpoint seperatrely

## Downloading what's required

In [ ]:
# Install dependencies (skip if already done)
import os

# Set environment variables
os.environ['CCD_MIRROR_PATH'] = ''
os.environ['PDB_MIRROR_PATH'] = ''

if not os.path.isfile("FOUNDRY_READY"):
    print("Installing rc-foundry...")

    # Uninstall torchvision first to avoid operator conflicts
    os.system("pip uninstall -y torchvision")

    # Install rc-foundry
    os.system("pip install -q 'rc-foundry[mpnn,rfd3]'")


    # Mark as ready
    os.system("touch FOUNDRY_READY")

    print("Done!")
else:
    print("rc-foundry already installed.")

Installing rc-foundry...
Done!


In [ ]:
# Download model weights (skips already-downloaded models automatically)
# In total, ~6GB (3GB for RFD3, 3GB for RF3, <100MB for MPNN); may take a few minutes depending on your connection speed
os.system("foundry install ligandmpnn")

0

In [ ]:
import warnings
warnings.filterwarnings('ignore', module='atomworks')

# Shared utilities for visualization (from AtomWorks)
from atomworks.io.utils.visualize import view

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "user": "Session.username",


In [ ]:
!mkdir /content/rfd3_checkpoint/
!foundry install rfd3 --checkpoint-dir /content/rfd3_checkpoint/
!mkdir /content/rfd3_unmodified_scytalone_theozyme_output

Install target: /content/rfd3_checkpoint

Installing rfd3: RFdiffusion3 checkpoint
✓ Successfully installed rfd3 to /content/rfd3_checkpoint/rfd3_latest.ckpt

Installation complete!


In [ ]:
!git clone https://github.com/RosettaCommons/foundry.git

Cloning into 'foundry'...
remote: Enumerating objects: 11438, done.
remote: Counting objects: 100% (964/964), done.
remote: Compressing objects: 100% (336/336), done.
remote: Total 11438 (delta 732), reused 651 (delta 625), pack-reused 10474 (from 3)
Receiving objects: 100% (11438/11438), 101.23 MiB | 17.11 MiB/s, done.
Resolving deltas: 100% (7295/7295), done.


### Now we have to unzip HBPLUS from our google drive

Read the user agreement for HBPLUS. This does not count as me sharing HBPLUS, since it is in my private drive, and I only worked on college premises, as agreed upon in the user agreement.

If you want to use HBPLUS you need obtain your own version of it

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%%bash

# 1. Unzip the downloaded file (the -q flag keeps the output quiet/clean)
unzip -q '/content/drive/MyDrive/embo_workshop_pes/hbplus.zip'

# 2. Go into the directory and compile the program
cd hbplus
make

# Note: If 'make' throws an error, uncomment the two lines below to use the manual fallback method from the instructions:
gcc -c *.c
gcc -o hbplus hbp_gen.o hbp_inpdb.o hbp_findh.o hbp_hhb.o hbp_qnh.o hbp_main.o -lm

# 3. Move the compiled executable to Colab's global path and make sure it has execution permissions
mv hbplus /usr/local/bin/
chmod +x /usr/local/bin/hbplus

echo "hbplus successfully installed to /usr/local/bin/"

gcc -c -O3 hbp_gen.c
gcc -c -O3 hbp_inpdb.c
gcc -c -O3 hbp_findh.c
gcc -c -O3 hbp_hhb.c 
gcc -c -O3 hbp_qnh.c 
gcc -c -O3 hbp_main.c
gcc -O3 -o hbplus \
hbp_gen.o hbp_inpdb.o hbp_findh.o hbp_hhb.o hbp_qnh.o hbp_main.o -lm
✅ hbplus successfully installed to /usr/local/bin/


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "user": "Session.username",
hbp_inpdb.c: In function ‘inpdb_file’:
hbp_inpdb.c:2529:17: warning: ignoring return value of ‘fgets’ declared with attribute ‘warn_unused_result’ [-Wunused-result]
 2529 |                 fgets(sstbuf,160,sstfp);
      |                 ^~~~~~~~~~~~~~~~~~~~~~~
In file included from /usr/include/stdio.h:894,
                 from hbp_inpdb.c:73:
In function ‘fgets’,
    inlined from ‘inpdb_file’ at hbp_inpdb.c:2356:14:
/usr/include/x86_64-linux-gnu/bits/stdio2.h:268:12: warning: call to ‘__fgets_chk_warn’ declared with attribute warning: fgets called with bigger size than length of destination buffer [-Wattribute-warning]
  268 |     return __fgets_chk_warn (__s, sz, __n, __stream);
    

In [ ]:
# This creates the .env file (if it doesn't exist) and appends the path to it
!echo "HBPLUS_PATH=/usr/local/bin/hbplus" >> /content/foundry/.env

# Let's read the file just to verify it was written correctly
!cat /content/foundry/.env

# USAGE INSTRUCTIONS:
# - copy this file and rename it to `.env` to use it.
# - do not commit the `.env` file to version control. 
# - you will need to set the paths below as appropriate for your environment.
#   We provide examples in the comments to give you an idea of the expected format.

# --- Mirrors to RCSB data ---

# The `PDB_MIRROR_PATH` is a path to a local mirror of the PDB database. It's 
# expected that you use the same saving conventions as the RCSB PDB, which means:
#   `1a2b` --> /path/to/pdb_mirror/a2/1a2b.cif.gz
# To set up a mirror, you can use tha atomworks commandline: `atomworks pdb sync /path/to/mirror`
PDB_MIRROR_PATH=

# The `CCD_MIRROR_PATH` is a path to a local mirror of the CCD database.
# It's expected that you use the same saving conventions as the RCSB CCD, which means:
#  `HEM` --> /path/to/ccd_mirror/H/HEM/HEM.cif
# To set up a mirror, you can use the atomworks commandline: `atomworks ccd sync /path/to/mirror`
# If no mirror is provided, the internal b

## RFD3 below, based on documentation instead of given google colab notebook

### Running RFD3 from below

In [ ]:
%%writefile /content/rfd3_scytalone_input_cfg_config.json
{
  "scytalone_theozyme_unmodified": {
    "input": "/content/proper_scytalone_theozyme_sorted.pdb",
    "unindex": "A1,A2,A3",
    "length": "200",
    "ligand": "UNK",
    "plddt_enhanced": true,
    "select_fixed_atoms": {
      "A1-3": "ALL",
      "UNK": "ALL"
    },
    "select_hbond_donor": {
      "A1": "ND1",
      "A3": "NZ"
    },
    "select_hbond_acceptor": {
      "A2": "OD1,OD2"
    }
  }
}

Overwriting /content/rfd3_scytalone_input_partial_diffusion_config.json


Do not give length if using partial diffusion. Otherwise, the syntax is the same as rest

Use the below code for partial_diffusion, where you'll have to give the whole protein, and not just the theozyme

In [ ]:
%%writefile /content/rfd3_scytalone_input_partial_diffusion_30A_config.json

{
  "scytalone_theozyme_partial_diffusion": {
    "input": "/content/sorted_rfd3_promising_proper_ligand_scaff1.pdb",
    "unindex": "A93,A34,A106",
    "ligand": "UNK",
    "plddt_enhanced": true,
    "partial_t": 30.0,
    "select_fixed_atoms": {
      "A106": "ALL",
      "A34": "ALL",
      "A93": "ALL",
      "UNK": "ALL"
    },
    "select_hbond_donor": {
      "A106": "ND1",
      "A34": "NZ"
    },
    "select_hbond_acceptor": {
      "A93": "OD1,OD2"
    }
  }
}

Writing /content/rfd3_scytalone_input_partial_diffusion_30A_config.json


The below code is just to run the demo. Need not be run if confident that RFD3 is working as intended

In [ ]:
#!rfd3 design out_dir=/content/rfd3_demo_output inputs=/content/foundry/models/rfd3/docs/examples/demo.json skip_existing=False dump_trajectories=True prevalidate_inputs=True ckpt_path="/content/rfd3_checkpoint/rfd3_latest.ckpt"

/usr/local/lib/python3.12/dist-packages/hydra/_internal/config_loader_impl.py:216: UserWarning: provider=hydra.searchpath in main, path=configs is not available.
  warnings.warn(
Environment variable CCD_MIRROR_PATH not set. Will not be able to use function requiring this variable. To set it you may:
  (1) add the line 'export VAR_NAME=path/to/variable' to your .bashrc or .zshrc file
  (2) set it in your current shell with 'export VAR_NAME=path/to/variable'
  (3) write it to a .env file in the root of the atomworks.io repository
Environment variable PDB_MIRROR_PATH not set. Will not be able to use function requiring this variable. To set it you may:
  (1) add the line 'export VAR_NAME=path/to/variable' to your .bashrc or .zshrc file
  (2) set it in your current shell with 'export VAR_NAME=path/to/variable'
  (3) write it to a .env file in the root of the atomworks.io repository
06:25:52 DEBUG transforms: Debug mode is on
06:25:53 INFO rfd3.engine: [rank: 0] Outputs will be written to /

In [ ]:
!rfd3 design \
    out_dir=/content/rfd3_scytalone_input_partialdiff_30A_try2 \
    inputs=/content/rfd3_scytalone_input_partial_diffusion_30A_config.json\
    n_batches=1 \
    diffusion_batch_size=5 \
    ckpt_path=/content/rfd3_checkpoint/rfd3_latest.ckpt \
    prevalidate_inputs=True

/usr/local/lib/python3.12/dist-packages/hydra/_internal/config_loader_impl.py:216: UserWarning: provider=hydra.searchpath in main, path=configs is not available.
  warnings.warn(
16:21:05 DEBUG transforms: Debug mode is on
16:21:07 INFO rfd3.engine: [rank: 0] Outputs will be written to /content/rfd3_scytalone_input_partialdiff_30A_try2.
16:21:07 INFO rfd3.engine: [rank: 0] Prevalidating design specification for example: rfd3_scytalone_input_partial_diffusion_30A_config_scytalone_theozyme_partial_diffusion
16:21:07 WARNING atomworks.io: We can't fix formal charges without building from templates, as we need to know the true number of hydrogens bonded to a given atom, not the inferred number. This may lead to occasional inaccuracies after adding inter-residue bonds. To avoid this and fix formal charges, set `add_missing_atoms = True`.
16:21:08 INFO rfd3.engine: [rank: 0] Found 0 existing example IDs in the output directory (0 total).
Using bfloat16 Automatic Mixed Precision (AMP)
16:21:2

To process the generated .cif.gz files into .cif, .pdb, and .fa files for convience, use the code below. It was generated with Gemini.

In [ ]:
import os
import glob
import gzip
import shutil
import biotite.structure.io.pdbx as pdbx
import biotite.structure.io.pdb as pdb
from biotite.structure import get_residue_starts
from biotite.sequence import ProteinSequence

def process_rfd3_outputs(folder_path):
    """
    Unzips .cif.gz files, extracts the first frame to a .pdb file,
    and natively extracts the FASTA sequence using Biotite.
    """
    print(f"Scanning folder: {folder_path}...\n")

    # 1. Unzip any .cif.gz files first
    gz_files = glob.glob(os.path.join(folder_path, '*.cif.gz'))
    for gz_path in gz_files:
        cif_path = gz_path.replace('.gz', '')
        with gzip.open(gz_path, 'rb') as f_in:
            with open(cif_path, 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
        os.remove(gz_path) # Delete the original .gz archive
        print(f"Unzipped: {os.path.basename(cif_path)}")

    # 2. Find all .cif files
    cif_files = glob.glob(os.path.join(folder_path, '*.cif'))
    if not cif_files:
        print("No .cif files found to process.")
        return

    print(f"\nFound {len(cif_files)} CIF files. Extracting frames and sequences...")

    for input_cif in cif_files:
        base_name = input_cif.replace('.cif', '')
        output_pdb = f"{base_name}.pdb"
        output_fasta = f"{base_name}.fa"

        # --- A. Read the CIF and Extract Frame 1 ---
        cif_file = pdbx.CIFFile.read(input_cif)
        first_frame_array = pdbx.get_structure(cif_file, model=1)

        # Write PDB
        pdb_out = pdb.PDBFile()
        pdb.set_structure(pdb_out, first_frame_array)
        pdb_out.write(output_pdb)

        # --- B. Extract Sequence using Biotite's native tools ---
        # Instantly get the indices for the start of every unique residue
        res_starts = get_residue_starts(first_frame_array)
        unique_residues = first_frame_array[res_starts]

        fasta_entries = {}

        for atom in unique_residues:
            chain = atom.chain_id
            res_name = atom.res_name

            if chain not in fasta_entries:
                fasta_entries[chain] = []

            # Safely attempt to convert to 1-letter code
            try:
                seq_1letter = ProteinSequence.convert_letter_3to1(res_name)
            except KeyError:
                # If it's a ligand (like SCY/UNK) or solvent, label it as 'X'
                seq_1letter = 'X'

            fasta_entries[chain].append(seq_1letter)

        # Write FASTA file, separated by chain
        with open(output_fasta, 'w') as f:
            for chain_id, seq_list in fasta_entries.items():
                sequence = "".join(seq_list)
                f.write(f">{os.path.basename(base_name)}_Chain_{chain_id}\n")
                f.write(f"{sequence}\n")

        print(f"✅ Generated: {os.path.basename(output_pdb)} & .fa")

    print("\n🎉 All files processed successfully!")

In [ ]:
process_rfd3_outputs("/content/rfd3_scytalone_input_partialdiff_30A_try2")

Scanning folder: /content/rfd3_scytalone_input_partialdiff_30A_try2...

Unzipped: rfd3_scytalone_input_partial_diffusion_30A_config_scytalone_theozyme_partial_diffusion_0_model_0.cif
Unzipped: rfd3_scytalone_input_partial_diffusion_30A_config_scytalone_theozyme_partial_diffusion_0_model_1.cif
Unzipped: rfd3_scytalone_input_partial_diffusion_30A_config_scytalone_theozyme_partial_diffusion_0_model_2.cif
Unzipped: rfd3_scytalone_input_partial_diffusion_30A_config_scytalone_theozyme_partial_diffusion_0_model_3.cif
Unzipped: rfd3_scytalone_input_partial_diffusion_30A_config_scytalone_theozyme_partial_diffusion_0_model_4.cif

Found 5 CIF files. Extracting frames and sequences...
✅ Generated: rfd3_scytalone_input_partial_diffusion_30A_config_scytalone_theozyme_partial_diffusion_0_model_3.pdb & .fa
✅ Generated: rfd3_scytalone_input_partial_diffusion_30A_config_scytalone_theozyme_partial_diffusion_0_model_0.pdb & .fa
✅ Generated: rfd3_scytalone_input_partial_diffusion_30A_config_scytalone_theoz

In [ ]:
import shutil
import os
from google.colab import files

def download(folder_name, drive_path, folder_path):

    # 1. Define paths
    folder_to_download = folder_path
    output_zip_file = f'/content/{folder_name}.zip'

    # 2. Compress the directory ONCE into Colab's local storage
    base_name = output_zip_file.replace('.zip', '')
    shutil.make_archive(base_name, 'zip', folder_to_download)
    print(f"✅ Successfully zipped '{folder_to_download}' into '{output_zip_file}'")

    # 3. Copy the finished zip file directly to your Google Drive
    try:
        shutil.copy(output_zip_file, drive_path)
        print(f"✅ Successfully backed up to Drive at: '{drive_path}'")
    except Exception as e:
        print(f"❌ Failed to copy to Drive. Did you mount it? Error: {e}")

    # 4. Trigger the browser download
    files.download(output_zip_file)

In [ ]:
download("rfd3_scytalone_input_partialdiff_30A_try2", "/content/drive/MyDrive/embo_workshop_pes", "/content/rfd3_scytalone_input_partialdiff_30A_try2")

✅ Successfully zipped '/content/rfd3_scytalone_input_partialdiff_30A_try2' into '/content/rfd3_scytalone_input_partialdiff_30A_try2.zip'
✅ Successfully backed up to Drive at: '/content/drive/MyDrive/embo_workshop_pes'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Pre-ligandMPNN

In [ ]:
import biotite.structure.io.pdb as pdb

# 1. Define the path to your PDB file (make sure it's uploaded to your Colab environment)
pdb_path = "/content/rfd3_scytalone_theozyme_trp_modified_config_scytalone_theozyme_trp_modified_0_model_3.pdb" #optimization for model3 in trp_try7

# 2. Read the file into a Biotite PDBFile object
pdb_file = pdb.PDBFile.read(pdb_path)
protein_array = pdb_file.get_structure(model=1)

# 3. Load your separate ligand AtomArray
#ligand_file = pdb.PDBFile.read("/content/proper_ligand_alone.pdb")
#ligand_array = ligand_file.get_structure(model=1)

# 4. Combine them into a single complex!
#complex_array = protein_array + ligand_array

# Now you can visualize it exactly like you did in your screenshot!
view(protein_array)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
print(protein_array) #this is for checking some stuff

    A       1  SER N      N        -5.647   24.436   11.847
    A       1  SER CA     C        -5.918   23.602   10.657
    A       1  SER C      C        -5.792   22.185   10.906
    A       1  SER O      O        -5.171   21.687   11.857
    A       1  SER CB     C        -4.952   24.065    9.586
    A       1  SER OG     O        -3.646   23.563    9.840
    A       2  ALA N      N        -6.415   21.306   10.033
    A       2  ALA CA     C        -6.332   19.827   10.171
    A       2  ALA C      C        -4.907   19.336   10.100
    A       2  ALA O      O        -4.517   18.438   10.843
    A       2  ALA CB     C        -7.203   19.189    9.093
    A       3  LEU N      N        -4.108   19.936    9.219
    A       3  LEU CA     C        -2.711   19.569    9.096
    A       3  LEU C      C        -1.931   19.936   10.347
    A       3  LEU O      O        -1.111   19.065   10.803
    A       3  LEU CB     C        -2.087   20.293    7.889
    A       3  LEU CG     C        -0.62

---

## Section 2: Sequence Design with MPNN

Protein and Ligand MPNN (Message Passing Neural Network) designs amino acid sequences that will fold into a target backbone structure.

**Model Options:**
- `protein_mpnn`: Original ProteinMPNN for protein-only design
- `ligand_mpnn`: Extended model supporting ligand-aware design

**Key Parameters:**
- `batch_size`: Number of sequences to generate per structure
- `remove_waters`: Whether to exclude water molecules from context

In [ ]:
from mpnn.inference_engines.mpnn import MPNNInferenceEngine

# Configure MPNN inference engine
# See mpnn.utils.inference.MPNN_GLOBAL_INFERENCE_DEFAULTS for all options
engine_config = {
    "model_type": "ligand_mpnn",  # or "protein_mpnn" for vanilla ProteinMPNN
    "is_legacy_weights": True,    # Required for now for ligand_mpnn and protein_mpnn
    "out_directory": '/content/ligandmpnn_rfd3_trp_sundae_try6_model3',
    "write_structures": True,
    "write_fasta": True,
}

# Configure per-input inference options
# See mpnn.utils.inference.MPNN_PER_INPUT_INFERENCE_DEFAULTS for all options
input_configs = [
    {
        "batch_size": 50,         # Generate 50 sequences per structure
        "remove_waters": True,
        "temperature": 0.05,
        "fixed_residues": ['A24','A28','A114', 'A148','B205'], #B1 is the ligand
        "ligand_mpnn_use_side_chain_context": 1, #22 in https://github.com/dauparas/LigandMPNN?tab=readme-ov-file#design-examples
        "ligand_mpnn_cutoff_for_score": 5.0, #32 in https://github.com/dauparas/LigandMPNN?tab=readme-ov-file#design-examples
        "save_stats": 1,
        "pack_side_chains": 1,
        "number_of_packs_per_design": 1,
        "pack_with_ligand_context": 1 #1 in https://github.com/dauparas/LigandMPNN?tab=readme-ov-file#side-chain-packing-examples
    }
]

# Run sequence design on the RFD3-generated backbone
model = MPNNInferenceEngine(**engine_config)
mpnn_outputs = model.run(input_dicts=input_configs, atom_arrays=[protein_array])

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "user": "Session.username",


In [ ]:
from mpnn.inference_engines.mpnn import MPNNInferenceEngine
from biotite.structure import get_residue_starts
from biotite.sequence import ProteinSequence

# Extract and display the designed sequences
print(f"Generated {len(mpnn_outputs)} designed sequences:\n")

for i, item in enumerate(mpnn_outputs):
    res_starts = get_residue_starts(item.atom_array)
    # Convert 3-letter codes to 1-letter using Biotite
    seq_1letter = ''.join(
        ProteinSequence.convert_letter_3to1(res_name)
        for res_name in item.atom_array.res_name[res_starts]
    )
    print(f"Sequence {i+1}: {seq_1letter}")

Generated 50 designed sequences:

Sequence 1: MVLVAALERDMDIDELLRAAAELGFRHVSLWPSEGIPISEMPRVKAEAAKYGLEAVYAITGSTPGIGSSSAPRDAAQIRAAAEEARALGITKVMIHIDTGTQDLSVVPMAVREARAAGLEVRFLDNGGNADATPAEVAARVRTMLAIREELGLDTPVLAKGLRDAAQARAARAAGADGILLGSLVVRDPERAREIIAAAR
Sequence 2: MVLVASLERDMDIEELCRAAAALGFTSITLWPSHGIPISQLPTIAATAKKYGLEAVYAITGSTPGIGSSSEPRNAAQIAAAAAQAKALGITKVTIHIDTGTKDPAVVPTAVRVARAAGLEVAFIDNGGGATLTPEEIAERVRLMREIARELGLDAPIIAKGLRNAEQARAAREAGADGINLGRLIVRDPERAREIIAAAR
Sequence 3: MELVGALMPDMDIDEHLAAAAALGFRTVTLWPSYGWPLSLAPRVKAAAAKYGLEAVYSITGADPGIGSSSRPRDAAQIAAAAAAAKALGITKVDIHIDTRTQPLEVVPMAYRVATAAGLEVAFIDNGGGGTATPAEIAAAVRKAREIAAELGSDAPIIAKGLRDAAQARAAREAGADGILLGRLIVDDPARAREIIAAAR
Sequence 4: MKLVACLERDMDIAEHLRAAAELGFTDVTLWPSEGWPISELPAVKAEAAKYGLRAVYSITGSDPGIGSSSRPRNAEEIRAAAREAKALGITEVDIHIDTGTEDLSVVPMAVREARAEGLEVRYLDNGGGSTLSPEEIAERVRTALEIRRELGLDTPVIAKGLRNAEQARAAVEAGADGIALGRLVVDDPERAREIIAAAK
Sequence 5: MVLVACLERDMDIAELLRAAAELGFTAVTLWPSEGWPISEAPRVQAEAAKYGLEAIYSITGADPGIGSESRPRDAAQIAAAAREAKALGITKVDIHVDTHTQ